# 第四阶段：STL（标准模板库）

## 实验 7：`std::variant` —— 显式表达有限状态集合

实验 6 的 `std::optional<T>` 表达“一个 `T` 或没有值”。当结果可能是多个结构不同、但都合法的状态时，可以使用 `std::variant<A, B, C>`。

variant 在任一时刻只包含一个 active alternative，并负责该对象的构造与析构。本实验关注：

- 如何构造、查询和安全访问 active alternative；
- `std::visit` 如何集中并穷尽处理状态；
- alternative 切换时，contained object 的生命周期如何变化；
- `monostate`、重复类型和 `valueless_by_exception` 的边界；
- 如何把 C++ variant 映射为 tagged C ABI 与 Kotlin sealed 类型。

In [ ]:
// 本步骤：引入 variant、解析、字符串、断言和 ABI 示例所需的标准库。
#include <cassert>
#include <charconv>
#include <cstddef>
#include <cstdint>
#include <iostream>
#include <optional>
#include <string>
#include <string_view>
#include <system_error>
#include <utility>
#include <variant>

### 1. variant 是类型安全的 tagged union

下面的解析结果只能是成功值或解析错误：

```text
ParseResult
    │
    ├── ParsedNumber { value }
    │
    └── ParseError { code, message }
```

variant 同时维护 active alternative 标记和足以容纳最大 alternative 的存储。与 C union 不同，标准库负责构造、销毁和访问当前有效成员；但调用方仍要正确处理每一种业务状态。

In [ ]:
// 本步骤：定义成功与失败两种不同类型，并组合成封闭的解析结果。
struct ParsedNumber
{
    int value;
};

struct ParseError
{
    // 用枚举保留机器可判断的原因，message 负责可读诊断。
    enum class Code
    {
        empty_input,
        invalid_format,
        out_of_range
    };

    Code code;
    std::string message;
};

using ParseResult =
    std::variant<ParsedNumber, ParseError>;

ParseResult parse_number(std::string_view text)
{
    // 空输入是明确的解析错误，不尝试访问 data()。
    if (text.empty())
    {
        return ParseError{
            ParseError::Code::empty_input,
            "input is empty"};
    }

    int value = 0;

    // from_chars 不分配内存，并返回停止位置与错误码。
    const auto [end, error] =
        std::from_chars(
            text.data(),
            text.data() + text.size(),
            value);

    // 数值超出 int 范围时保留独立错误原因。
    if (error == std::errc::result_out_of_range)
    {
        return ParseError{
            ParseError::Code::out_of_range,
            "number is out of range"};
    }

    // 转换失败或没有消费完整输入，都视为格式错误。
    if (error != std::errc{} ||
        end != text.data() + text.size())
    {
        return ParseError{
            ParseError::Code::invalid_format,
            "invalid integer"};
    }

    // 成功 alternative 使用独立结构体，避免与错误字段混淆。
    return ParsedNumber{value};
}

### 2. 构造并查询 active alternative

从某个 alternative 的值构造 variant 时，该类型成为 active alternative。`holds_alternative<T>()` 按类型查询状态；`index()` 返回 alternative 在类型列表中的位置。

业务逻辑优先按类型查询，不要把 `index() == 0` 等数字写进协议或持久化数据。调整 variant 的类型顺序会改变 index，而且其对象布局也不是稳定 ABI。

In [ ]:
// 本步骤：构造成功和失败结果，并按类型检查当前 active alternative。
{
    const ParseResult success = parse_number("42");
    const ParseResult failure = parse_number("42ms");

    // holds_alternative 不读取 payload，只检查当前状态类型。
    assert(std::holds_alternative<ParsedNumber>(success));
    assert(std::holds_alternative<ParseError>(failure));

    // index 仅用于本地观察类型顺序，不作为业务协议。
    std::cout << "success index = " << success.index() << '\n';
    std::cout << "failure index = " << failure.index() << '\n';

    assert(success.index() == 0);
    assert(failure.index() == 1);
}

### 3. `get_if` 与 `get` 使用不同失败策略

- `std::get_if<T>(&value)`：类型匹配时返回指向 contained object 的借用指针，否则返回 null；
- `std::get<T>(value)`：类型匹配时返回引用，否则抛出 `std::bad_variant_access`；
- `std::holds_alternative<T>()`：只检查，不访问。

从 `get_if` 或 `get` 得到的 pointer/reference 不拥有对象。variant 切换 alternative 或析构后，旧借用立即失效。

In [ ]:
// 本步骤：对同一个结果分别演示无异常检查和抛异常访问。
{
    ParseResult result = parse_number("123");

    // get_if 安全探测成功类型，并借用其中的 ParsedNumber。
    const ParsedNumber *number =
        std::get_if<ParsedNumber>(&result);
    assert(number != nullptr);
    assert(number->value == 123);

    // 不匹配的 get_if 返回 null，不会抛异常。
    assert(std::get_if<ParseError>(&result) == nullptr);

    // get 适用于调用方已确定类型的场景；类型错误会抛异常。
    try
    {
        static_cast<void>(std::get<ParseError>(result));
    }
    catch (const std::bad_variant_access &error)
    {
        std::cout << "caught: " << error.what() << '\n';
    }
}

### 4. 使用 `std::visit` 集中处理所有状态

连续写多个 `get_if` 适合简单分支；状态增多时，`std::visit` 可以把每个 alternative 的处理放在一个 visitor 中。

下面的 `Overloaded` 把多个 lambda 合并为一个 visitor。若新增 alternative 却没有对应 overload，编译器会报告 visitor 无法处理该类型。这种编译期反馈很适合封闭状态模型。

In [ ]:
// 本步骤：定义组合多个 lambda 的通用 visitor 辅助类型。
template<class... Visitors>
struct Overloaded : Visitors...
{
    // 把每个 lambda 的 operator() 都引入当前重载集合。
    using Visitors::operator()...;
};

template<class... Visitors>
Overloaded(Visitors...) -> Overloaded<Visitors...>;

In [ ]:
// 本步骤：用一个穷尽 visitor 分别处理成功值和解析错误。
{
    const ParseResult success = parse_number("42");
    const ParseResult failure = parse_number("abc");

    const auto print_result = [](const ParseResult &result)
    {
        // 每个 lambda 精确对应一种 alternative。
        std::visit(
            Overloaded{
                [](const ParsedNumber &number)
                {
                    std::cout
                        << "value = "
                        << number.value
                        << '\n';
                },
                [](const ParseError &error)
                {
                    std::cout
                        << "error = "
                        << error.message
                        << '\n';
                }},
            result);
    };

    // 同一个 visitor 根据 active alternative 自动分派。
    print_result(success);
    print_result(failure);
}

### 5. variant 拥有 active alternative

variant 内部的 active object 由 variant 自己拥有。调用 `emplace<T>()` 或赋值为另一 alternative 时，旧对象先结束生命周期，再在同一 variant 存储中构造新对象。

下面通过构造和析构日志观察状态切换。

In [ ]:
// 本步骤：定义带生命周期日志的 payload，用于观察 alternative 切换。
class TrackedPayload
{
public:
    explicit TrackedPayload(std::string label)
        : label_(std::move(label))
    {
        // 构造日志表示 payload 成为 active alternative。
        std::cout << label_ << " constructed\n";
    }

    ~TrackedPayload() noexcept
    {
        // 析构日志表示 variant 正在离开或切换该 alternative。
        std::cout << label_ << " destroyed\n";
    }

    const std::string &label() const noexcept
    {
        // 返回 contained string 的只读借用。
        return label_;
    }

private:
    std::string label_;
};

In [ ]:
// 本步骤：切换 active alternative，并在失效前后重新获取借用。
{
    std::variant<std::monostate, TrackedPayload> state;

    // 默认状态是第一个可默认构造的 monostate。
    assert(std::holds_alternative<std::monostate>(state));

    // 直接构造 payload，并临时借用其内部 label。
    state.emplace<TrackedPayload>("running");
    const std::string *borrowed =
        &std::get<TrackedPayload>(state).label();
    assert(*borrowed == "running");

    // 切回 monostate 会析构 payload；此后绝不能读取旧 borrowed。
    state.emplace<std::monostate>();

    // 只有重新进入 payload 状态后，才能重新取得合法借用。
    state.emplace<TrackedPayload>("finished");
    borrowed = &std::get<TrackedPayload>(state).label();
    assert(*borrowed == "finished");
}

### 6. `monostate` 提供显式空闲状态

variant 默认构造时会默认构造第一个 alternative。如果第一个类型不能默认构造，variant 自身也不能默认构造。

`std::monostate` 是一个可默认构造、没有业务数据的占位类型。把它放在第一位，可以显式表达“尚未开始”“空闲”等合法状态。它不同于 `valueless_by_exception`：monostate 是设计好的业务状态。

In [ ]:
// 本步骤：定义任务的空闲、运行和失败状态。
struct Running
{
    int task_id;
};

struct Failed
{
    std::string message;
};

using TaskState =
    std::variant<std::monostate, Running, Failed>;

In [ ]:
// 本步骤：在三个合法任务状态之间切换并读取各自 payload。
{
    TaskState state;

    // 默认构造得到显式的空闲状态。
    assert(std::holds_alternative<std::monostate>(state));

    // emplace 进入运行状态，并直接构造 task_id。
    state.emplace<Running>(Running{7});
    assert(std::get<Running>(state).task_id == 7);

    // 赋值为 Failed 会先销毁 Running，再构造错误信息。
    state = Failed{"network unavailable"};
    assert(
        std::get<Failed>(state).message ==
        "network unavailable");
}

### 7. alternative 类型应清晰且唯一

`std::variant<int, int>` 在语法上允许，但从单个 `int` 构造和 `get<int>()` 都无法判断目标位置，只能使用下标形式，接口容易出错。

优先为不同含义建立命名类型：

```cpp
struct Width  { int value; };
struct Height { int value; };

using Dimension = std::variant<Width, Height>;
```

命名类型让 visitor、日志和协议映射都更清晰。除非确实按位置建模，否则避免重复 alternative 类型。

### 8. optional、variant 与错误结果的选择

| 需求 | 更合适的类型 |
| --- | --- |
| `T` 或正常缺失 | `optional<T>` |
| 多个并列、合法状态 | `variant<A, B, C>` |
| 明确的成功值或错误值 | `expected<T, E>`（工具链支持时）或专用 result |
| 跨 C ABI 的结果 | status/tag + C payload |

`variant<Success, Error>` 可以实现结果类型，但 variant 本身不规定哪个 alternative 是成功，也没有专用的错误传播语义。类型选择应反映 API 契约，而不只是“能够装下这些值”。

### 9. C ABI 使用显式 tag 和 C payload

`std::variant` 是 C++ 模板类型，布局、alternative index 和异常行为都不是稳定 C ABI。跨边界时应设计固定宽度的 tag，并规定每个 tag 下哪些字段有效。直接暴露普通 `enum` 还可能受底层宽度影响，因此本实验使用 `int32_t` tag 常量。

本实验的 C 结果只包含一个固定宽度整数 payload；错误通过 tag 表达。调用方必须先读取 tag，只有 `SDK_PARSE_OK` 时才能把 value 当作成功结果。

In [ ]:
// 本步骤：定义稳定的 C tag/result，并把 C++ ParseResult 映射到边界类型。
using sdk_parse_tag = std::int32_t;

// 固定每个公开 tag 的数值，避免依赖 C++ variant index 或 enum 宽度。
inline constexpr sdk_parse_tag SDK_PARSE_OK = 0;
inline constexpr sdk_parse_tag SDK_PARSE_EMPTY = 1;
inline constexpr sdk_parse_tag SDK_PARSE_INVALID_FORMAT = 2;
inline constexpr sdk_parse_tag SDK_PARSE_OUT_OF_RANGE = 3;
inline constexpr sdk_parse_tag SDK_PARSE_INVALID_ARGUMENT = 4;
inline constexpr sdk_parse_tag SDK_PARSE_INTERNAL_ERROR = 5;

struct sdk_parse_result
{
    sdk_parse_tag tag;
    std::int32_t value;
};

sdk_parse_tag map_parse_error(
    ParseError::Code code) noexcept
{
    // 将每个 C++ 错误 alternative 映射为固定 C tag。
    switch (code)
    {
    case ParseError::Code::empty_input:
        return SDK_PARSE_EMPTY;
    case ParseError::Code::invalid_format:
        return SDK_PARSE_INVALID_FORMAT;
    case ParseError::Code::out_of_range:
        return SDK_PARSE_OUT_OF_RANGE;
    }

    // 防御未来遗漏的枚举值，避免把未知状态映射成成功。
    return SDK_PARSE_INTERNAL_ERROR;
}

extern "C" sdk_parse_result sdk_parse_number(
    const char *data,
    std::size_t size) noexcept
{
    // 非零长度必须对应有效输入地址；空输入可直接映射。
    if (data == nullptr)
    {
        return size == 0
            ? sdk_parse_result{SDK_PARSE_EMPTY, 0}
            : sdk_parse_result{SDK_PARSE_INVALID_ARGUMENT, 0};
    }

    try
    {
        // string_view 和 ParseResult 都只存在于 C++ 实现内部。
        const ParseResult result =
            parse_number(std::string_view(data, size));

        if (const ParsedNumber *number =
                std::get_if<ParsedNumber>(&result))
        {
            return sdk_parse_result{
                SDK_PARSE_OK,
                static_cast<std::int32_t>(number->value)};
        }

        const ParseError &error =
            std::get<ParseError>(result);

        return sdk_parse_result{
            map_parse_error(error.code),
            0};
    }
    catch (...)
    {
        // 捕获所有 C++ 异常，保证它们不会穿过 C ABI。
        return sdk_parse_result{
            SDK_PARSE_INTERNAL_ERROR,
            0};
    }
}

In [ ]:
// 本步骤：验证成功、格式错误和无效参数三种稳定边界结果。
{
    constexpr char valid_text[] = "42";

    // pointer + length 只在同步调用期间借用输入字符。
    const sdk_parse_result success =
        sdk_parse_number(valid_text, 2);
    assert(success.tag == SDK_PARSE_OK);
    assert(success.value == 42);

    constexpr char invalid_text[] = "4x";
    const sdk_parse_result failure =
        sdk_parse_number(invalid_text, 2);

    // 失败时只读取 tag，不把 value 当作业务结果。
    assert(failure.tag == SDK_PARSE_INVALID_FORMAT);

    const sdk_parse_result invalid_argument =
        sdk_parse_number(nullptr, 1);
    assert(
        invalid_argument.tag ==
        SDK_PARSE_INVALID_ARGUMENT);
}

### 10. Kotlin wrapper 映射为 sealed 类型

Kotlin 没有与 `std::variant` ABI 兼容的对象。wrapper 应读取 C tag，再构造惯用的 sealed class/interface：

```kotlin
sealed interface ParseResult {
    data class Success(val value: Int) : ParseResult
    data class Error(val reason: ParseError) : ParseResult
}
```

映射时必须穷尽所有 C tag。未知 tag 应作为协议错误处理，而不是默认归入某个旧状态。C++ variant、C tag 和 Kotlin sealed hierarchy 是三层不同表示，它们共享的是状态语义，不是内存布局。

### 11. `valueless_by_exception` 与布局边界

variant 切换 alternative 时，如果构造新对象抛出且无法保留旧对象，variant 可能进入 `valueless_by_exception()` 状态。此时 `index() == std::variant_npos`，对它调用 `visit` 或错误的 `get` 会抛出 `bad_variant_access`。

常规值类型和 noexcept move 可以降低出现该状态的机会，但通用基础设施仍应知道它存在。不要把它当作业务上的“空状态”；需要合法空状态时使用 `monostate`。

variant 的大小、对齐、tag 表示和 alternative 排列均属实现细节，不能用 `memcpy` 序列化，也不能直接跨编译器或跨语言 ABI。

In [ ]:
// 本步骤：检查普通结果保持有效 alternative，不依赖具体 index 编码。
{
    const ParseResult result = parse_number("7");

    // 正常构造不会进入异常造成的无值状态。
    assert(!result.valueless_by_exception());
    assert(result.index() != std::variant_npos);

    // 按类型读取比把 index 数字传播到业务层更稳定。
    const ParsedNumber *number =
        std::get_if<ParsedNumber>(&result);
    assert(number != nullptr);
    assert(number->value == 7);
}

### 12. 使用 variant 前的检查清单

- 状态集合是否有限、互斥，并且每个状态都需要不同数据？
- alternative 是否使用清晰的命名类型，避免重复基础类型？
- 访问时选择了 `get_if`、已验证的 `get`，还是穷尽的 `visit`？
- 从 contained object 取得的 pointer/reference 是否跨过了状态切换或 variant 析构？
- 是否需要合法默认状态？若需要，是否显式使用 `monostate`？
- 这是并列状态，还是更适合 optional/expected 的缺失或成功-失败模型？
- C ABI 是否使用稳定 tag、固定宽度 payload，并捕获了所有 C++ 异常？
- Kotlin wrapper 是否穷尽映射 tag，并拒绝未知协议状态？

### 本实验结论

`std::variant<A, B, C>` 是拥有型的封闭状态容器：任一时刻只构造一个 active alternative，并在切换或析构时正确结束其生命周期。它比手工 union 更安全，但调用方仍必须穷尽处理业务状态。

`get_if` 提供无异常探测，`get` 在类型错误时抛异常，`visit` 将各 alternative 的处理集中到编译期重载集合。contained object 的借用不能跨越 emplace、赋值或 variant 析构。

variant 不能直接跨 C ABI。稳定边界应使用显式 tag 与 C-compatible payload，并捕获 C++ 异常；Kotlin wrapper 再把 tag 映射为 sealed 类型。三层表示共享状态含义，但绝不共享对象布局。